<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%203/Aula_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3.3 — Refinando o time + RAG no Olheiro

**Curso:** Agno: Criando agentes e sistemas multiagente

**Aula:** 3 — Times de IA: como fazer agentes trabalharem juntos

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 3.2** o time ganhou o **Analista** com tools de visualização e ativamos memória de sessão.  
- Treinador-líder
- Olheiro
- Analista (tools de visualização e memória de sessão).  

Porém, o Olheiro -> **fontes públicas** (Tavily, Wikipedia).

Plano da aula:

1. Adicionar **RAG (Retrieval-Augmented Generation)** ao Olheiro com Knowledge do Agno
2. Carregar uma **base interna** com regras oficiais da Copa do Mundo 2026
3. Refinar **regras de delegação** do Treinador-líder
4. Demo final com a pergunta-âncora exercitando todas as capacidades



## Setup

Adicionamos `lancedb` (banco vetorial local).


In [11]:
!pip install -U agno openai tavily-python wikipedia plotly lancedb tantivy pylance

In [12]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — Boa prática: um modelo por agente

Isso parece redundante hoje, mas é um padrão que traz ganhos:

- **Em produção**
- **Em debugging**
- **Em comparação A/B**



In [13]:
from agno.models.openai import OpenAIChat

modelo_olheiro   = OpenAIChat(id="gpt-5.4") # add temperature
modelo_analista  = OpenAIChat(id="gpt-5.4")
modelo_treinador = OpenAIChat(id="gpt-5.4")

---

## Passo 2 — Por que RAG?

| Tipo | Exemplo no contexto da Seleção |
|---|---|
| **Documentos oficiais estruturados** | Regulamento da Copa do Mundo, regras da CBF, manuais técnicos |
| **Bases internas do clube** | Relatórios de scouting, observações da comissão técnica, histórico tático |
| **Documentação proprietária** | Contratos, dados médicos, decisões internas |





---

## Passo 3 — Construindo a base interna

**Regras oficiais da Copa do Mundo de 2026**.

O texto cobre:

- **Formato da Copa**
- **Critérios de desempate**
- **Regras das fases eliminatórias**


In [14]:
regras_copa_2026 = [
    """
    REGULAMENTO COPA DO MUNDO FIFA 2026 — FORMATO DA COMPETIÇÃO

    A Copa do Mundo de 2026 será sediada nos Estados Unidos, Canadá e México,
    com início em 11 de junho de 2026 e final em 19 de julho de 2026 em East Rutherford, New Jersey.

    Esta é a primeira Copa com 48 seleções participantes (anteriormente eram 32).
    As 48 equipes são distribuídas em 12 grupos de 4 times cada.

    Cada equipe joga 3 partidas na fase de grupos (round-robin), com sistema padrão de pontuação:
    - 3 pontos por vitória
    - 1 ponto por empate
    - 0 ponto por derrota

    Avançam para a fase eliminatória:
    - Os 12 primeiros colocados de cada grupo
    - Os 12 segundos colocados de cada grupo
    - Os 8 melhores terceiros colocados (ranqueados entre todos os 12 grupos)

    Total: 32 seleções avançam para a fase eliminatória, criando uma rodada inédita
    chamada "Round of 32" antes das oitavas de final.

    Para garantir integridade esportiva, a última rodada de cada grupo deve ser jogada
    no mesmo dia e mesmo horário.
    """,

    """
    REGULAMENTO COPA DO MUNDO FIFA 2026 — CRITÉRIOS DE DESEMPATE NA FASE DE GRUPOS

    Quando duas ou mais equipes terminam a fase de grupos com a mesma pontuação,
    a FIFA aplica os seguintes critérios em ordem estrita:

    Primeiro nível (apenas entre equipes empatadas — confronto direto):
    1. Maior número de pontos obtidos nos jogos entre as equipes empatadas
    2. Melhor saldo de gols nos jogos entre as equipes empatadas
    3. Maior número de gols marcados nos jogos entre as equipes empatadas

    Segundo nível (todas as partidas do grupo):
    4. Melhor saldo de gols em todas as partidas do grupo
    5. Maior número de gols marcados em todas as partidas do grupo

    Terceiro nível (disciplina e ranking):
    6. Melhor pontuação de fair play (cartões: amarelo = -1, vermelho indireto = -3,
       vermelho direto = -4, amarelo + vermelho direto = -5)
    7. Posição no ranking mundial atual da FIFA
    8. Sorteio (último recurso)

    Para classificação dos 8 melhores terceiros colocados (regra exclusiva de 2026):
    Os 12 terceiros colocados são ranqueados em uma única tabela considerando:
    pontos, saldo de gols, gols marcados, fair play e ranking FIFA — nessa ordem.
    Os 8 com melhor desempenho avançam para o Round of 32.

    NOTA HISTÓRICA: O critério de fair play foi usado uma única vez em Copas do Mundo,
    em 2018, quando o Japão eliminou Senegal por ter recebido menos cartões amarelos.
    """,

    """
    REGULAMENTO COPA DO MUNDO FIFA 2026 — FASE ELIMINATÓRIA

    A fase eliminatória é disputada em 5 rodadas:
    1. Round of 32 (32 seleções) — rodada inédita devido à expansão para 48 times
    2. Oitavas de final (16 seleções)
    3. Quartas de final (8 seleções)
    4. Semifinais (4 seleções)
    5. Final (2 seleções)

    A disputa de terceiro lugar acontece em 18 de julho de 2026, em Miami.
    A final é em 19 de julho de 2026, no MetLife Stadium (East Rutherford, NJ).

    Em caso de empate na fase eliminatória:
    - Disputa-se prorrogação de 2 tempos de 15 minutos
    - Persistindo o empate, decisão por pênaltis

    Substituições por jogo:
    - Até 5 substituições por equipe
    - Realizadas em até 3 paradas (mais o intervalo)
    - Em jogos com prorrogação, é permitida 1 substituição extra (totalizando 6)

    Pausas hidratantes obrigatórias:
    - 3 minutos no meio de cada tempo
    - Devido às altas temperaturas previstas em sedes como México, Houston e Dallas

    Cartões acumulados:
    - 2 cartões amarelos em jogos diferentes geram suspensão automática para o próximo jogo
    - Os cartões amarelos zeram após as quartas de final (não carregam para semifinais)
    """,
]

print(f"Base interna construída com {len(regras_copa_2026)} documentos.")

Base interna construída com 3 documentos.


---

## Passo 4 — Plugando a base no Agno

https://docs.agno.com/knowledge/concepts/readers/overview


In [15]:
from agno.knowledge.knowledge import Knowledge
from agno.knowledge.embedder.openai import OpenAIEmbedder
from agno.vectordb.lancedb import LanceDb, SearchType

# Vector DB local — sem servidor, sem cadastro
vector_db = LanceDb(
    table_name="regras_copa_2026",
    uri="/tmp/lancedb_scoutai",
    search_type=SearchType.hybrid,
    embedder=OpenAIEmbedder(id="text-embedding-3-small"),
)

knowledge = Knowledge(
    name="Regras Oficiais Copa 2026",
    description="Base interna com formato, critérios de desempate e regras das fases eliminatórias da Copa do Mundo 2026.",
    vector_db=vector_db,
)

# Carregando os documentos na base. Pode demorar alguns segundos
# (chama o embedder pra cada documento e armazena no vector DB).
for i, conteudo in enumerate(regras_copa_2026):
    knowledge.add_content(text_content=conteudo, name=f"regulamento_2026_parte_{i+1}")

print("Base de conhecimento carregada e indexada.")

INFO Adding content from regulamento_2026_parte_1

INFO Selecting reader for extension: Text

INFO Deleted 1 records with content_hash 'b697b1a8b8a98f797f3b4450d1b02a02b87923f734bceeca82498b87868be26f' from   
     table 'regras_copa_2026'.

INFO Adding content from regulamento_2026_parte_2

INFO Selecting reader for extension: Text

INFO Deleted 1 records with content_hash 'eebbce13cde7135e6615d2f1b6a0696279f77d4d880e605c007383a0443ba578' from   
     table 'regras_copa_2026'.

INFO Adding content from regulamento_2026_parte_3

INFO Selecting reader for extension: Text

INFO Deleted 1 records with content_hash '747eb2b50d62d3427d610b1e5ddbf1aee9dc2a6bbf81e19d399ded31f9c727aa' from   
     table 'regras_copa_2026'.

Base de conhecimento carregada e indexada.


---

## Passo 5 — Recriando o Olheiro com acesso à base

- Tavily (dados recentes),
- Wikipedia (dados históricos)
- Base interna (regras oficiais).


In [16]:
from agno.agent import Agent
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas e em base interna oficial",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável.",
        "Você tem três fontes disponíveis:",
        "• BASE INTERNA (Knowledge): regulamento oficial da Copa do Mundo 2026 — formato, critérios de desempate, regras de fase eliminatória.",
        "• Tavily (web): eventos recentes — últimos jogos, convocações, forma de jogadores.",
        "• Wikipedia: fatos históricos consolidados — Copas antigas, biografias, técnicos do passado.",
        "Sempre que a pergunta envolver REGRAS OFICIAIS da Copa 2026, busque PRIMEIRO na base interna — é a fonte autoritativa.",
        "Para forma atual, use Tavily. Para histórico, use Wikipedia. Combine fontes quando a pergunta tiver múltiplas necessidades.",
        "Retorne dados objetivos — não interprete nem opine.",
        "Se a busca falhar em todas as fontes, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    knowledge=knowledge,            # ← peça nova: base interna como knowledge
    search_knowledge=True,          # ← permite ao agente buscar na base
    markdown=True,
)

---

## Passo 6 — Validando o RAG

> *"Quais são os critérios de desempate na fase de grupos da Copa de 2026, em ordem?"*



In [17]:
olheiro.print_response(
    "Quais são os critérios de desempate na fase de grupos da Copa de 2026, em ordem?",
    stream=True,
)

Output()

INFO Found 3 documents

---

## Passo 7 — Refinando regras de delegação

Recriando o Analista e o time com Treinador-líder refinado:


In [18]:
# Recriando as tools do Analista (mesma definição da Aula 3.2)
import plotly.graph_objects as go
from agno.tools import tool

@tool(
    name="comparar_em_barras",
    description=(
        "Gera gráfico de barras verticais comparando jogadores em uma métrica única. "
        "Use quando o usuário pedir comparação direta entre jogadores em um único atributo "
        "(ex: gols, assistências, finalizações). "
        "Retorna confirmação de que o gráfico foi exibido."
    ),
)
def comparar_em_barras(
    jogadores: list[str],
    valores: list[float],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de barras interativo comparando jogadores."""
    fig = go.Figure(data=[go.Bar(x=jogadores, y=valores)])
    fig.update_layout(
        title=titulo,
        xaxis_title="Jogador",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de barras '{titulo}' exibido para o usuário."

def evolucao_em_linhas(
    jogadores: list[str],
    periodos: list[str],
    valores_por_jogador: list[list[float]],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de linhas interativo mostrando evolução temporal."""
    fig = go.Figure()
    for nome, valores in zip(jogadores, valores_por_jogador):
        fig.add_trace(go.Scatter(x=periodos, y=valores, mode="lines+markers", name=nome))
    fig.update_layout(
        title=titulo,
        xaxis_title="Período",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de linhas '{titulo}' exibido para o usuário."

In [19]:
analista = Agent(
    name="Analista",
    role="Analista de desempenho que interpreta dados e gera visualizações táticas",
    model=modelo_analista,
    instructions=[
        "Você é o Analista de desempenho do ScoutAI FC.",
        "Sua função é receber dados (geralmente do Olheiro) e produzir análise visual.",
        "Use `comparar_em_barras` para comparações diretas entre jogadores em uma métrica.",
        "Use `evolucao_em_linhas` para progressões ao longo do tempo.",
        "Sempre que gerar um gráfico, acompanhe de uma análise interpretativa de 2-3 frases "
        "destacando o que o gráfico revela.",
        "Quando receber pedido de gráfico ou visualização, "
        "SEMPRE chame `comparar_em_barras` ou `evolucao_em_linhas` — nunca produza ASCII, "
        "Mermaid ou tabelas como substituto. As tools são o entregável correto.",
        "Se receber dados insuficientes ou ambíguos, peça esclarecimento em vez de inventar números.",
    ],
    tools=[comparar_em_barras, evolucao_em_linhas],
    markdown=True,
)

1. **Anti-redundância**: depois que o Olheiro trouxe dados, o Treinador não pede ao Analista pra buscar de novo
2. **Parsimônia**: para perguntas conceituais simples, responder direto sem delegar (economiza latência e custo)
3. **Uso prioritário da base interna**: quando a pergunta envolve regras oficiais, a base interna é a primeira escolha

In [20]:
from agno.team import Team

time_scoutai = Team(
    name="Treinador do ScoutAI FC",
    mode="coordinate",
    members=[olheiro, analista],
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC, líder da comissão técnica.",
        "Você coordena o Olheiro (busca dados, incluindo base interna oficial) "
        "e o Analista (interpreta dados e gera gráficos).",

        # Regras de delegação refinadas
        "REGRAS DE DELEGAÇÃO:",
        "• Para REGRAS OFICIAIS da Copa 2026 (formato, desempate, fases eliminatórias), "
        "delegue ao Olheiro — ele consultará a base interna.",
        "• Para EVENTOS RECENTES ou FATOS HISTÓRICOS, delegue ao Olheiro — ele escolhe entre Tavily e Wikipedia.",
        "• Para COMPARAÇÃO VISUAL ou GRÁFICO, garanta que o Olheiro trouxe os dados primeiro, "
        "então delegue ao Analista APENAS pedindo o gráfico, não peça ao Analista pra buscar dados.",
        "O Analista tem tools próprias de Plotly e escolherá a apropriada.",
        "NÃO especifique formato (Mermaid, ASCII, tabela) — confie nas tools do Analista.",
        "• Para PERGUNTAS CONCEITUAIS SIMPLES (esquemas táticos consagrados, regras gerais do futebol) ",
        "responda direto sem delegar — economiza latência e custo.",
        "• Sempre integre dados e análise antes de responder ao usuário, em português do Brasil.",
    ],
    add_history_to_context=True,
    num_history_runs=3,
    markdown=True,
)

---

## Passo 8 — Pergunta-âncora exercitando todas as capacidades


> *"O Brasil está numa fase de grupos disputada. Considerando os critérios oficiais de desempate da Copa 2026, e dada a forma atual dos atacantes brasileiros, qual seria o cenário mais arriscado para a Seleção?"*

O fluxo esperado:

1. Treinador identifica três necessidades distintas
2. Delega ao **Olheiro** as duas buscas: regras oficiais (base interna) + forma dos atacantes (Tavily)
3. Recebe os dados e faz análise tática integrada (interpretação)
4. Responde ao usuário em português, integrando tudo


In [21]:
time_scoutai.print_response(
    "O Brasil está numa fase de grupos disputada. "
    "Considerando os critérios oficiais de desempate da Copa 2026, "
    "e dada a forma atual dos atacantes brasileiros, "
    "qual seria o cenário mais arriscado para a Seleção?",
    session_id="aula-3-3",
    stream=True,
)

WARNING  add_history_to_context is True, but no database has been assigned to the team. History will not be added  
         to the context.

Output()

INFO Found 3 documents

---

## Os três modos de Team no Agno

| Modo | Como funciona | Quando usar |
|---|---|---|
| **`route`** | Líder funciona como roteador — escolhe **um** especialista e repassa a pergunta inteira pra ele. Não integra nada. | Quando os especialistas são mutuamente exclusivos e a resposta vem direta de um deles (ex: "perguntas técnicas → suporte; perguntas comerciais → vendas") |
| **`coordinate`** | Líder decide quem chamar, **delega tarefas específicas**, recebe resultados, **integra** numa resposta final. | Quando a pergunta exige combinação de competências distintas (nosso caso — Olheiro busca, Analista visualiza, Treinador integra) |
| **`collaborate`** | Todos os especialistas recebem a pergunta **em paralelo**, cada um responde da sua perspectiva. Líder agrega as visões. | Quando você quer **múltiplas perspectivas independentes** sobre a mesma pergunta (ex: análise de risco com financeiro + jurídico + operacional respondendo em paralelo) |



---

## O custo de ter um time


- **Latência multiplica.**

- **Custo por turno cresce.**

- **Pontos de falha aumentam.**



## Quando time vale a pena

| Situação | Time vale? |
|---|---|
| Pergunta complexa que combina dado externo + análise + visualização | ✅ Sim — exatamente o caso |
| Pergunta conceitual simples ("regras gerais", "explicação tática") | ❌ Não — agente único é mais rápido e barato |
| Domínio com fontes muito heterogêneas (web + base interna + APIs) | ✅ Sim — separar Olheiros por fonte se necessário |
| Volume muito alto e perguntas repetitivas | ⚠️ Depende — vale considerar cache em vez de mais agentes |
| Protótipo sendo validado | ❌ Comece com agente único e evolua |

>Time é **decisão de arquitetura**, não obrigação.


---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time completo de 3 agentes especializados
│   ├── Treinador (líder com regras de delegação refinadas)
│   ├── Olheiro (Tavily + Wikipedia + Knowledge/RAG)
│   └── Analista (comparar_em_barras + evolucao_em_linhas)
├── ✅ Memória de sessão (conversa flui entre turnos)
├── ✅ Modelos declarados por agente (boa prática para produção)
│
├── ❌ Sem fluxo determinístico para tarefas estruturadas  → Aula 4 (Workflows)
└── ❌ Sem produção / monitoramento / governança          → Aula 5 (AgentOS)
```

### Aplicando em outros contextos

| Domínio | Estrutura equivalente |
|---|---|
| **Suporte técnico SaaS** | Triagem (líder) + Buscador (docs internas via RAG) + Resolvedor (passo-a-passo) |
| **Atendimento jurídico** | Recepcionista (líder) + Pesquisador (base de jurisprudência via RAG) + Redator (minutas) |
| **Análise financeira** | Estrategista (líder) + Coletor (cotações + relatórios via RAG) + Analista (gráficos) |
| **RH (recrutamento)** | Recrutador (líder) + Buscador (CVs internos via RAG + LinkedIn) + Avaliador (ranking) |


### Próxima aula

**Aula 4 — Orquestrando agentes com workflows**

Na Aula 4 vamos transformar parte do que o time faz hoje em **pipeline determinístico**, e introduzir o **Auxiliar Técnico** — quarto e último agente do produto.
